# Análise de Dados — Camada Gold

Este notebook apresenta as análises realizadas a partir das tabelas da camada Gold, com o objetivo de responder às perguntas de negócio definidas para o MVP.

As análises utilizam o modelo analítico construído nas etapas anteriores do pipeline, explorando informações relacionadas a vendas, produtos, clientes, pagamentos e experiência dos clientes.

# Problema de Negócio
O presente projeto busca compreender a evolução e as características das vendas realizadas por vendedores integrados ao ecossistema Olist em diferentes marketplaces brasileiros, identificando padrões relacionados ao desempenho comercial, às categorias de produtos, à distribuição geográfica e ao comportamento de compra dos clientes, além de explorar aspectos relacionados à operação logística e à experiência do consumidor.

## Pergunta Central
Como os aspectos comerciais, geográficos e operacionais caracterizam os pedidos realizados no ecossistema Olist e de que forma o desempenho logístico se relaciona com a experiência dos clientes?

Para responder ao problema central, a análise será estruturada em cinco dimensões:

- Comercial: 
Como o volume de pedidos e o valor vendido evoluíram ao longo do período analisado?

- Produtos: 
Quais categorias de produtos apresentam maior participação no valor vendido e no volume de vendas, e como se diferenciam em relação ao ticket médio?

- Clientes: 
Como os clientes e pedidos estão distribuídos geograficamente e qual é o comportamento de recompra?

- Pagamentos: 
Quais são as principais formas de pagamento utilizadas nos pedidos e como os clientes utilizam parcelamento e múltiplos meios de pagamento?

- Experiência: 
Pedidos entregues após a data estimada apresentam avaliações diferentes daqueles entregues dentro ou antes do prazo?

### **1. Comercial**: 
Como o volume de pedidos e o valor vendido evoluíram ao longo do período analisado?

#### 1.1 Definição do escopo da análise

Antes da análise da evolução comercial, foi avaliada a distribuição dos pedidos por status para definir quais registros devem compor as métricas de volume e valor vendido.

Essa validação permite verificar a representatividade de pedidos não concluídos antes da aplicação de eventuais filtros na análise.

In [0]:
%sql
-- Avalia a distribuição de pedidos e valores por status para definição do escopo da análise comercial.
SELECT
    status_pedido,
    COUNT(*) AS qtd_pedidos,
    SUM(qtd_itens) AS qtd_itens,
    ROUND(SUM(valor_produtos), 2) AS valor_produtos,
    ROUND(SUM(valor_total_pedido), 2) AS valor_total_pedido
FROM workspace.gold.ft_pedidos
GROUP BY status_pedido
ORDER BY qtd_pedidos DESC;

status_pedido,qtd_pedidos,qtd_itens,valor_produtos,valor_total_pedido
delivered,96478,110197,13221498.11,15419773.75
shipped,1107,1185,150727.44,177129.34
canceled,625,542,95235.27,105885.72
unavailable,609,7,2007.69,2140.49
invoiced,314,359,61526.37,68988.75
processing,301,357,60439.22,69394.11
created,5,0,0.00,0.00
approved,2,3,209.60,241.08


In [0]:
%sql
-- Calcula a evolução mensal do volume de pedidos, itens e valor dos produtos associados à demanda.
SELECT
    date(date_trunc('month', data_compra)) AS date_month,
    COUNT(*) AS qtd_pedidos,
    ROUND(SUM(valor_produtos), 2) AS valor_pedidos,
    ROUND(SUM(valor_produtos) / COUNT(*), 2) AS valor_medio_pedido
FROM workspace.gold.ft_pedidos
WHERE date(data_compra) BETWEEN '2017-01-01' and '2018-08-01' -- filtro para trazer apenas datas onde os dados estão corretos para analise
GROUP BY 1
ORDER BY 1

date_month,qtd_pedidos,valor_pedidos,valor_medio_pedido
2017-01-01,800,120312.87,150.39
2017-02-01,1780,247303.02,138.93
2017-03-01,2682,374344.30,139.58
2017-04-01,2404,359927.23,149.72
2017-05-01,3700,506071.14,136.78
2017-06-01,3245,433038.60,133.45
2017-07-01,4026,498031.48,123.70
2017-08-01,4331,573971.68,132.53
2017-09-01,4285,624401.69,145.72
2017-10-01,4631,664219.43,143.43


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Calcula mensalmente a participação de pedidos entregues e cancelados sobre o total de pedidos.
SELECT
    date(DATE_TRUNC('MONTH', data_compra)) AS date_month,
    COUNT(*) AS qtd_pedidos,
    SUM(CASE WHEN status_pedido = 'delivered' THEN 1 ELSE 0 END) AS qtd_entregues,
    SUM(CASE WHEN status_pedido = 'canceled' THEN 1 ELSE 0 END) AS qtd_cancelados,
    ROUND(
        100.0 * SUM(CASE WHEN status_pedido = 'canceled' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS taxa_cancelamento_pct
FROM workspace.gold.ft_pedidos
WHERE date(data_compra) BETWEEN '2017-01-01' and '2018-08-01' -- filtro para trazer apenas datas onde os dados estão corretos para analise
GROUP BY 1
ORDER BY 1

date_month,qtd_pedidos,qtd_entregues,qtd_cancelados,taxa_cancelamento_pct
2017-01-01,800,750,3,0.38
2017-02-01,1780,1653,17,0.96
2017-03-01,2682,2546,33,1.23
2017-04-01,2404,2303,18,0.75
2017-05-01,3700,3546,29,0.78
2017-06-01,3245,3135,16,0.49
2017-07-01,4026,3872,28,0.70
2017-08-01,4331,4193,27,0.62
2017-09-01,4285,4150,20,0.47
2017-10-01,4631,4478,26,0.56


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Analisa o volume diário de pedidos em novembro de 2017 para investigar o pico associado ao período da Black Friday.
SELECT
    data_compra,
    COUNT(*) AS qtd_pedidos,
    ROUND(SUM(valor_produtos), 2) AS valor_pedidos
FROM workspace.gold.ft_pedidos
WHERE data_compra >= '2017-11-01'
  AND data_compra < '2017-12-01'
GROUP BY data_compra
ORDER BY data_compra;

data_compra,qtd_pedidos,valor_pedidos
2017-11-01,111,19383.61
2017-11-02,124,21263.73
2017-11-03,143,22844.05
2017-11-04,111,19169.07
2017-11-05,144,17327.44
2017-11-06,193,28600.29
2017-11-07,160,20829.71
2017-11-08,175,19240.51
2017-11-09,191,25053.34
2017-11-10,165,20223.84



### **2. Produtos**:  
Quais categorias de produtos apresentam maior participação no valor vendido e no volume de vendas, e como se diferenciam em relação ao ticket médio?

#### 2.1 Definição do escopo da análise

Esta análise busca identificar quais categorias de produtos possuem maior participação no desempenho comercial da plataforma, considerando tanto o **volume de itens vendidos** quanto o **valor vendido**.

Diferentemente da análise anterior, que avaliou a demanda registrada independentemente do status final do pedido, nesta etapa são considerados apenas os itens pertencentes a **pedidos entregues (`delivered`)**, de forma que as métricas representem vendas efetivamente concluídas.

O **volume de vendas** corresponde à quantidade de itens vendidos em cada categoria, enquanto o **valor vendido** corresponde à soma do valor dos produtos, desconsiderando o frete. Como métrica complementar, será analisado o **valor médio por item**, permitindo identificar diferenças no perfil de preço entre as categorias.

A análise será realizada a partir da tabela fato `ft_vendas`, relacionada à dimensão `dm_produto` para identificação das categorias e à `ft_pedidos` para seleção dos pedidos entregues.

In [0]:
%sql
-- Agrupa as 30 categorias com maior valor vendido e consolida as demais em Outros para facilitar a visualização.
WITH vendas_categoria AS (
    SELECT
        INITCAP(REPLACE(p.categoria_produto, '_', ' ')) AS categoria_produto,
        COUNT(*) AS qtd_itens_vendidos,
        SUM(v.valor_item) AS valor_vendido
    FROM workspace.gold.ft_vendas v
    INNER JOIN workspace.gold.dm_produto p
        ON v.id_produto = p.id_produto
    INNER JOIN workspace.gold.ft_pedidos pe
        ON v.id_pedido = pe.id_pedido
    WHERE pe.status_pedido = 'delivered'
    GROUP BY p.categoria_produto
),

ranking AS (
    SELECT
        *,
        ROW_NUMBER() OVER (ORDER BY valor_vendido DESC) AS ranking_valor
    FROM vendas_categoria
)

SELECT
    CASE
        WHEN ranking_valor <= 30 THEN categoria_produto
        ELSE 'Outros'
    END AS categoria_produto,
    SUM(qtd_itens_vendidos) AS qtd_itens_vendidos,
    ROUND(SUM(valor_vendido), 2) AS valor_vendido
FROM ranking
GROUP BY
    CASE
        WHEN ranking_valor <= 30 THEN categoria_produto
        ELSE 'Outros'
    END
ORDER BY valor_vendido DESC;

categoria_produto,qtd_itens_vendidos,valor_vendido
Beleza Saude,9465,1233131.72
Relogios Presentes,5859,1166176.98
Cama Mesa Banho,10953,1023434.76
Esporte Lazer,8431,954852.55
Informatica Acessorios,7644,888724.61
Outros,7296,864175.70
Moveis Decoracao,8160,711927.69
Utilidades Domesticas,6795,615628.69
Cool Stuff,3718,610204.10
Automotivo,4140,578966.65


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Agrupa as 30 categorias com maior valor vendido e consolida as demais em Outros para facilitar a visualização.
WITH vendas_categoria AS (
    SELECT
        INITCAP(REPLACE(p.categoria_produto, '_', ' ')) AS categoria_produto,
        COUNT(*) AS qtd_itens_vendidos,
        SUM(v.valor_item) AS valor_vendido
    FROM workspace.gold.ft_vendas v
    INNER JOIN workspace.gold.dm_produto p
        ON v.id_produto = p.id_produto
    INNER JOIN workspace.gold.ft_pedidos pe
        ON v.id_pedido = pe.id_pedido
    WHERE pe.status_pedido = 'delivered'
    GROUP BY p.categoria_produto
),

ranking AS (
    SELECT
        *,
        ROW_NUMBER() OVER (ORDER BY qtd_itens_vendidos DESC) AS ranking_valor
    FROM vendas_categoria
)

SELECT
    CASE
        WHEN ranking_valor <= 30 THEN categoria_produto
        ELSE 'Outros'
    END AS categoria_produto,
    SUM(qtd_itens_vendidos) AS qtd_itens_vendidos,
    ROUND(SUM(valor_vendido), 2) AS valor_vendido
FROM ranking
GROUP BY
    CASE
        WHEN ranking_valor <= 30 THEN categoria_produto
        ELSE 'Outros'
    END
ORDER BY qtd_itens_vendidos DESC;

categoria_produto,qtd_itens_vendidos,valor_vendido
Cama Mesa Banho,10953,1023434.76
Beleza Saude,9465,1233131.72
Esporte Lazer,8431,954852.55
Moveis Decoracao,8160,711927.69
Informatica Acessorios,7644,888724.61
Utilidades Domesticas,6795,615628.69
Outros,6401,1119995.89
Relogios Presentes,5859,1166176.98
Telefonia,4430,309860.23
Ferramentas Jardim,4268,470495.28


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Compara participação em volume, participação em valor e valor médio por item entre as categorias de produtos.
WITH vendas_categoria AS (
    SELECT
        INITCAP(REPLACE(p.categoria_produto, '_', ' ')) AS categoria_produto,
        COUNT(*) AS qtd_itens_vendidos,
        SUM(v.valor_item) AS valor_vendido,
        AVG(v.valor_item) AS valor_medio_item
    FROM workspace.gold.ft_vendas v
    INNER JOIN workspace.gold.dm_produto p
        ON v.id_produto = p.id_produto
    INNER JOIN workspace.gold.ft_pedidos pe
        ON v.id_pedido = pe.id_pedido
    WHERE pe.status_pedido = 'delivered'
    GROUP BY p.categoria_produto
)

SELECT
    categoria_produto,
    qtd_itens_vendidos,
    ROUND(
        100.0 * qtd_itens_vendidos / SUM(qtd_itens_vendidos) OVER (),
        2
    ) AS participacao_volume_pct,
    ROUND(valor_vendido, 2) AS valor_vendido,
    ROUND(
        100.0 * valor_vendido / SUM(valor_vendido) OVER (),
        2
    ) AS participacao_valor_pct,
    ROUND(valor_medio_item, 2) AS valor_medio_item
FROM vendas_categoria
ORDER BY valor_vendido DESC
LIMIT 20

categoria_produto,qtd_itens_vendidos,participacao_volume_pct,valor_vendido,participacao_valor_pct,valor_medio_item
Beleza Saude,9465,8.59,1233131.72,9.33,130.28
Relogios Presentes,5859,5.32,1166176.98,8.82,199.04
Cama Mesa Banho,10953,9.94,1023434.76,7.74,93.44
Esporte Lazer,8431,7.65,954852.55,7.22,113.25
Informatica Acessorios,7644,6.94,888724.61,6.72,116.26
Moveis Decoracao,8160,7.40,711927.69,5.38,87.25
Utilidades Domesticas,6795,6.17,615628.69,4.66,90.60
Cool Stuff,3718,3.37,610204.10,4.62,164.12
Automotivo,4140,3.76,578966.65,4.38,139.85
Brinquedos,4030,3.66,471286.48,3.56,116.94


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Compara o valor médio por item entre as 10 categorias com maior valor vendido em pedidos entregues.
WITH vendas_categoria AS (
    SELECT
        INITCAP(REPLACE(p.categoria_produto, '_', ' ')) AS categoria_produto,
        COUNT(*) AS qtd_itens_vendidos,
        SUM(v.valor_item) AS valor_vendido,
        AVG(v.valor_item) AS valor_medio_item
    FROM workspace.gold.ft_vendas v
    INNER JOIN workspace.gold.dm_produto p
        ON v.id_produto = p.id_produto
    INNER JOIN workspace.gold.ft_pedidos pe
        ON v.id_pedido = pe.id_pedido
    WHERE pe.status_pedido = 'delivered'
    GROUP BY p.categoria_produto
),

ranking AS (
    SELECT
        *,
        ROW_NUMBER() OVER (ORDER BY valor_vendido DESC) AS ranking_valor
    FROM vendas_categoria
)

SELECT
    categoria_produto,
    ROUND(valor_vendido, 2) AS valor_vendido,
    ROUND(valor_medio_item, 2) AS valor_medio_item
FROM ranking
WHERE ranking_valor <= 10
ORDER BY valor_medio_item DESC;

categoria_produto,valor_vendido,valor_medio_item
Relogios Presentes,1166176.98,199.04
Cool Stuff,610204.10,164.12
Automotivo,578966.65,139.85
Beleza Saude,1233131.72,130.28
Brinquedos,471286.48,116.94
Informatica Acessorios,888724.61,116.26
Esporte Lazer,954852.55,113.25
Cama Mesa Banho,1023434.76,93.44
Utilidades Domesticas,615628.69,90.60
Moveis Decoracao,711927.69,87.25


Databricks visualization. Run in Databricks to view.

### ****3**. Clientes**:  
Como os clientes e pedidos estão distribuídos geograficamente e qual é o comportamento de recompra?

#### 3.1 Definição do escopo da análise

Esta análise busca compreender a distribuição geográfica dos clientes da plataforma e seu comportamento de recompra.

A localização será analisada a partir do estado associado a cada pedido, permitindo identificar as regiões com maior concentração de clientes e de pedidos.

Para a análise de recompra, será utilizado o `id_cliente_unico`, que permite reconhecer um mesmo consumidor em diferentes pedidos. Dessa forma, será possível comparar clientes que realizaram apenas uma compra com aqueles que realizaram mais de um pedido durante o período disponível na base.

Assim como na análise de produtos, serão considerados apenas pedidos entregues (`delivered`), de forma que a análise represente relações de compra efetivamente concluídas.

In [0]:
%sql
-- Analisa a distribuição de clientes e pedidos entregues por estado.
SELECT
    uf_cliente,
    COUNT(DISTINCT id_cliente_unico) AS qtd_clientes,
    COUNT(*) AS qtd_pedidos,
    ROUND(
        100.0 * COUNT(DISTINCT id_cliente_unico)
        / SUM(COUNT(DISTINCT id_cliente_unico)) OVER (),
        2
    ) AS participacao_clientes_pct,
    ROUND(
        100.0 * COUNT(*)
        / SUM(COUNT(*)) OVER (),
        2
    ) AS participacao_pedidos_pct
FROM workspace.gold.ft_pedidos
WHERE status_pedido = 'delivered'
GROUP BY uf_cliente
ORDER BY qtd_clientes DESC;

uf_cliente,qtd_clientes,qtd_pedidos,participacao_clientes_pct,participacao_pedidos_pct
SP,39156,40501,41.92,41.98
RJ,11917,12350,12.76,12.80
MG,11001,11354,11.78,11.77
RS,5168,5345,5.53,5.54
PR,4769,4923,5.11,5.10
SC,3449,3546,3.69,3.68
BA,3158,3256,3.38,3.37
DF,2019,2080,2.16,2.16
ES,1928,1995,2.06,2.07
GO,1895,1957,2.03,2.03


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Analisa a distribuição dos clientes pela quantidade de pedidos entregues realizados no período.
WITH pedidos_cliente AS (
    SELECT
        id_cliente_unico,
        COUNT(*) AS qtd_pedidos
    FROM workspace.gold.ft_pedidos
    WHERE status_pedido = 'delivered'
    GROUP BY id_cliente_unico
)

SELECT
    qtd_pedidos,
    COUNT(*) AS qtd_clientes,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS participacao_clientes_pct
FROM pedidos_cliente
GROUP BY qtd_pedidos
ORDER BY qtd_pedidos;

qtd_pedidos,qtd_clientes,participacao_clientes_pct
1,90557,97.00
2,2573,2.76
3,181,0.19
4,28,0.03
5,9,0.01
6,5,0.01
7,3,0.00
9,1,0.00
15,1,0.00


In [0]:
%sql
-- Compara a participação de clientes com compra única e clientes com recompra entre os pedidos entregues.
WITH pedidos_cliente AS (
    SELECT
        id_cliente_unico,
        COUNT(*) AS qtd_pedidos
    FROM workspace.gold.ft_pedidos
    WHERE status_pedido = 'delivered'
    GROUP BY id_cliente_unico
)

SELECT
    CASE
        WHEN qtd_pedidos = 1 THEN 'Compra única'
        ELSE 'Recompra'
    END AS perfil_cliente,
    COUNT(*) AS qtd_clientes,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS participacao_clientes_pct
FROM pedidos_cliente
GROUP BY
    CASE
        WHEN qtd_pedidos = 1 THEN 'Compra única'
        ELSE 'Recompra'
    END
ORDER BY qtd_clientes DESC;

perfil_cliente,qtd_clientes,participacao_clientes_pct
Compra única,90557,97.00
Recompra,2801,3.00


Databricks visualization. Run in Databricks to view.

### ****4**. Pagamentos**:  
Quais são as principais formas de pagamento utilizadas nos pedidos e como os clientes utilizam parcelamento e múltiplos meios de pagamento?

#### 4.1 Definição do escopo da análise

Esta análise busca compreender o comportamento de pagamento dos pedidos, identificando os meios de pagamento mais utilizados, o perfil de parcelamento e a utilização de múltiplos meios em uma mesma compra.

A análise utiliza a `ft_pagamentos`, cuja granularidade corresponde a um registro de pagamento dentro de um pedido. Como um mesmo pedido pode possuir mais de um registro de pagamento, as métricas são construídas respeitando essa característica da base.

Para manter consistência com as análises de vendas e clientes, são considerados apenas pagamentos associados a **pedidos entregues (`delivered`)**.

In [0]:
%sql
-- Analisa a participação dos meios de pagamento nos pedidos entregues e no valor total pago.
WITH pagamentos AS (
    SELECT
        pg.id_pedido,
        INITCAP(REPLACE(pg.tipo_pagamento, '_', ' ')) AS tipo_pagamento,
        pg.valor_pagamento
    FROM workspace.gold.ft_pagamentos pg
    INNER JOIN workspace.gold.ft_pedidos pe
        ON pg.id_pedido = pe.id_pedido
    WHERE pe.status_pedido = 'delivered'
),

total_pedidos AS (
    SELECT
        COUNT(DISTINCT id_pedido) AS total_pedidos
    FROM pagamentos
),

por_tipo AS (
    SELECT
        tipo_pagamento,
        COUNT(DISTINCT id_pedido) AS qtd_pedidos,
        SUM(valor_pagamento) AS valor_pagamentos
    FROM pagamentos
    GROUP BY tipo_pagamento
)

SELECT
    p.tipo_pagamento,
    p.qtd_pedidos,
    ROUND(
        100.0 * p.qtd_pedidos / t.total_pedidos,
        2
    ) AS participacao_pedidos_pct,
    ROUND(p.valor_pagamentos, 2) AS valor_pagamentos,
    ROUND(
        100.0 * p.valor_pagamentos / SUM(p.valor_pagamentos) OVER (),
        2
    ) AS participacao_valor_pct
FROM por_tipo p
CROSS JOIN total_pedidos t
ORDER BY p.qtd_pedidos DESC;

tipo_pagamento,qtd_pedidos,participacao_pedidos_pct,valor_pagamentos,participacao_valor_pct
Credit Card,74304,77.02,12101094.88,78.46
Boleto,19191,19.89,2769932.58,17.96
Voucher,3679,3.81,343013.19,2.22
Debit Card,1485,1.54,208421.12,1.35


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Analisa a distribuição do número de parcelas nos pagamentos com cartão de crédito de pedidos entregues.
SELECT
    pg.qtd_parcelas,
    COUNT(*) AS qtd_pagamentos,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS participacao_pagamentos_pct,
    ROUND(AVG(pg.valor_pagamento), 2) AS valor_medio_pagamento
FROM workspace.gold.ft_pagamentos pg
INNER JOIN workspace.gold.ft_pedidos pe
    ON pg.id_pedido = pe.id_pedido
WHERE pe.status_pedido = 'delivered'
  AND pg.tipo_pagamento = 'credit_card'
GROUP BY pg.qtd_parcelas
ORDER BY pg.qtd_parcelas;

qtd_parcelas,qtd_pagamentos,participacao_pagamentos_pct,valor_medio_pagamento
0,2,0.00,94.32
1,24759,33.20,95.60
2,12075,16.19,126.59
3,10164,13.63,142.00
4,6891,9.24,163.64
5,5095,6.83,182.30
6,3804,5.10,208.54
7,1563,2.10,185.95
8,4136,5.55,306.29
9,618,0.83,197.97


In [0]:
%sql
-- Analisa o parcelamento no cartão de crédito, agrupando pagamentos acima de 10 parcelas em uma única faixa.
WITH pagamentos_cartao AS (
    SELECT
        CASE
            WHEN pg.qtd_parcelas > 10 THEN '11+'
            ELSE CAST(pg.qtd_parcelas AS STRING)
        END AS faixa_parcelas,
        CASE
            WHEN pg.qtd_parcelas > 10 THEN 11
            ELSE pg.qtd_parcelas
        END AS ordem_parcelas,
        pg.valor_pagamento
    FROM workspace.gold.ft_pagamentos pg
    INNER JOIN workspace.gold.ft_pedidos pe
        ON pg.id_pedido = pe.id_pedido
    WHERE pe.status_pedido = 'delivered'
      AND pg.tipo_pagamento = 'credit_card'
)

SELECT
    faixa_parcelas,
    COUNT(*) AS qtd_pagamentos,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS participacao_pagamentos_pct,
    ROUND(AVG(valor_pagamento), 2) AS valor_medio_pagamento
FROM pagamentos_cartao
WHERE faixa_parcelas <> '0'
GROUP BY faixa_parcelas, ordem_parcelas
ORDER BY ordem_parcelas;

faixa_parcelas,qtd_pagamentos,participacao_pagamentos_pct,valor_medio_pagamento
1,24759,33.20,95.60
2,12075,16.19,126.59
3,10164,13.63,142.00
4,6891,9.24,163.64
5,5095,6.83,182.30
6,3804,5.10,208.54
7,1563,2.10,185.95
8,4136,5.55,306.29
9,618,0.83,197.97
10,5150,6.90,410.54


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Identifica a participação de pedidos entregues pagos com um ou múltiplos meios de pagamento.
WITH meios_por_pedido AS (
    SELECT
        pg.id_pedido,
        COUNT(DISTINCT pg.tipo_pagamento) AS qtd_meios_pagamento
    FROM workspace.gold.ft_pagamentos pg
    INNER JOIN workspace.gold.ft_pedidos pe
        ON pg.id_pedido = pe.id_pedido
    WHERE pe.status_pedido = 'delivered'
    GROUP BY pg.id_pedido
)

SELECT
    CASE
        WHEN qtd_meios_pagamento = 1 THEN 'Meio único'
        ELSE 'Múltiplos meios'
    END AS perfil_pagamento,
    COUNT(*) AS qtd_pedidos,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS participacao_pedidos_pct
FROM meios_por_pedido
GROUP BY
    CASE
        WHEN qtd_meios_pagamento = 1 THEN 'Meio único'
        ELSE 'Múltiplos meios'
    END
ORDER BY qtd_pedidos DESC;

perfil_pagamento,qtd_pedidos,participacao_pedidos_pct
Meio único,94295,97.74
Múltiplos meios,2182,2.26


Databricks visualization. Run in Databricks to view.

### ****5**. Experiência**:  
Pedidos entregues após a data estimada apresentam avaliações diferentes daqueles entregues dentro ou antes do prazo?

#### 5.1 Definição do escopo da análise

Esta análise busca avaliar se o cumprimento do prazo estimado de entrega está associado a diferenças na avaliação realizada pelos clientes.

São considerados apenas pedidos com status `delivered`, com data efetiva de entrega e data estimada de entrega disponíveis. Os pedidos são classificados em dois grupos:

- **No prazo ou antecipado:** entrega realizada na data estimada ou antes dela;
- **Atrasado:** entrega realizada após a data estimada.

A experiência do cliente é avaliada por meio da `nota_media_avaliacao` consolidada por pedido. Pedidos sem avaliação são preservados na camada Gold, porém não são considerados no cálculo das notas médias desta análise.

In [0]:
%sql
-- Compara volume de pedidos e nota média de avaliação entre entregas no prazo e atrasadas.
WITH entregas AS (
    SELECT
        id_pedido,
        CASE
            WHEN DATE(data_hora_entrega_cliente) <= data_estimada_entrega
                THEN 'No prazo ou antecipado'
            ELSE 'Atrasado'
        END AS status_prazo,
        nota_media_avaliacao
    FROM workspace.gold.ft_pedidos
    WHERE status_pedido = 'delivered'
      AND data_hora_entrega_cliente IS NOT NULL
      AND data_estimada_entrega IS NOT NULL
)

SELECT
    status_prazo,
    COUNT(*) AS qtd_pedidos,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS participacao_pedidos_pct,
    COUNT(nota_media_avaliacao) AS qtd_pedidos_avaliados,
    ROUND(AVG(nota_media_avaliacao), 2) AS nota_media
FROM entregas
GROUP BY status_prazo
ORDER BY qtd_pedidos DESC;

status_prazo,qtd_pedidos,participacao_pedidos_pct,qtd_pedidos_avaliados,nota_media
No prazo ou antecipado,89936,93.23,89443,4.29
Atrasado,6534,6.77,6381,2.27


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Compara a distribuição das notas de avaliação entre pedidos entregues no prazo e atrasados.
WITH entregas AS (
    SELECT
        id_pedido,
        CASE
            WHEN DATE(data_hora_entrega_cliente) <= data_estimada_entrega
                THEN 'No prazo ou antecipado'
            ELSE 'Atrasado'
        END AS status_prazo,
        nota_media_avaliacao
    FROM workspace.gold.ft_pedidos
    WHERE status_pedido = 'delivered'
      AND data_hora_entrega_cliente IS NOT NULL
      AND data_estimada_entrega IS NOT NULL
      AND nota_media_avaliacao IS NOT NULL
),

notas AS (
    SELECT
        status_prazo,
        ROUND(nota_media_avaliacao) AS nota,
        COUNT(*) AS qtd_pedidos
    FROM entregas
    GROUP BY
        status_prazo,
        ROUND(nota_media_avaliacao)
)

SELECT
    status_prazo,
    nota,
    qtd_pedidos,
    ROUND(
        100.0 * qtd_pedidos
        / SUM(qtd_pedidos) OVER (PARTITION BY status_prazo),
        2
    ) AS participacao_nota_pct
FROM notas
ORDER BY status_prazo, nota;

status_prazo,nota,qtd_pedidos,participacao_nota_pct
Atrasado,1.0,3426,53.69
Atrasado,2.0,553,8.67
Atrasado,3.0,695,10.89
Atrasado,4.0,652,10.22
Atrasado,5.0,1055,16.53
No prazo ou antecipado,1.0,5886,6.58
No prazo ou antecipado,2.0,2371,2.65
No prazo ou antecipado,3.0,7251,8.11
No prazo ou antecipado,4.0,18240,20.39
No prazo ou antecipado,5.0,55695,62.27


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Analisa a avaliação dos pedidos de acordo com a quantidade de dias de atraso na entrega.
WITH pedidos_atrasados AS (
    SELECT
        id_pedido,
        DATEDIFF(
            DATE(data_hora_entrega_cliente),
            data_estimada_entrega
        ) AS dias_atraso,
        nota_media_avaliacao
    FROM workspace.gold.ft_pedidos
    WHERE status_pedido = 'delivered'
      AND data_hora_entrega_cliente IS NOT NULL
      AND data_estimada_entrega IS NOT NULL
      AND nota_media_avaliacao IS NOT NULL
      AND DATE(data_hora_entrega_cliente) > data_estimada_entrega
),

faixas AS (
    SELECT
        *,
        CASE
            WHEN dias_atraso BETWEEN 1 AND 3 THEN '1 a 3 dias'
            WHEN dias_atraso BETWEEN 4 AND 7 THEN '4 a 7 dias'
            WHEN dias_atraso BETWEEN 8 AND 14 THEN '8 a 14 dias'
            WHEN dias_atraso BETWEEN 15 AND 30 THEN '15 a 30 dias'
            ELSE 'Mais de 30 dias'
        END AS faixa_atraso,
        CASE
            WHEN dias_atraso BETWEEN 1 AND 3 THEN 1
            WHEN dias_atraso BETWEEN 4 AND 7 THEN 2
            WHEN dias_atraso BETWEEN 8 AND 14 THEN 3
            WHEN dias_atraso BETWEEN 15 AND 30 THEN 4
            ELSE 5
        END AS ordem_faixa
    FROM pedidos_atrasados
)

SELECT
    faixa_atraso,
    COUNT(*) AS qtd_pedidos,
    ROUND(AVG(nota_media_avaliacao), 2) AS nota_media
FROM faixas
GROUP BY faixa_atraso, ordem_faixa
ORDER BY ordem_faixa;

faixa_atraso,qtd_pedidos,nota_media
1 a 3 dias,1852,3.29
4 a 7 dias,1748,2.11
8 a 14 dias,1446,1.67
15 a 30 dias,1006,1.62
Mais de 30 dias,329,2.06
